# Tabula Rasa Quickstart

A blank-slate AI that learns arithmetic from scratch, one specialist at a time.

This notebook demonstrates: loading a trained specialist, running inference,
inspecting the scratchpad carry propagation, and visualizing the transformer's
attention patterns.

In [ ]:
import sys, os
from pathlib import Path

# Add project source to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

import torch
from tabula_rasa.config import Config
from tabula_rasa.tokenizer import MathTokenizer
from tabula_rasa.model import MathTransformer, count_parameters
from tabula_rasa.generate import generate_answer

## 1. Load a Trained Specialist

We'll load the addition specialist. The model is a 1M-parameter causal
transformer with 4 layers, 4 heads, and 128-dim embeddings, trained on
carry-scratchpad arithmetic.

In [ ]:
# Configuration
cfg = Config()
tok = MathTokenizer()
cfg.vocab_size = tok.vocab_size

# Load checkpoint
checkpoint_path = Path('specialists/math/add/best.pt')
if not checkpoint_path.exists():
    # Try general specialist
    checkpoint_path = Path('specialists/math/general/best.pt')
    
if checkpoint_path.exists():
    model = MathTransformer(cfg)
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    sd = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(sd, strict=False)
    model.eval()
    global_step = ckpt.get('global_step', '?')
    print(f'Loaded: {checkpoint_path.name}')
    print(f'Params: {count_parameters(model):,}')
    print(f'Steps: {global_step}')
    print(f'Architecture: d={cfg.d_model}, L={cfg.n_layers}, H={cfg.n_heads}')
else:
    print('No trained checkpoint found. Run training first:')
    print('  python3 train_specialist.py add --quick')

## 2. Run Inference

Generate answers for arithmetic problems. The model uses a V2 scratchpad
format: combined carry-digit tokens like `04` mean carry=0, digit=4.

In [ ]:
def test_problem(prompt):
    """Run a single arithmetic problem and parse the scratchpad result."""
    output = model.generate(tok, prompt, max_new_tokens=20, temperature=0.0)
    
    # Parse scratchpad
    if '=' in output:
        raw = output.split('=', 1)[-1].replace('<EOS>', '').replace('<PAD>', '').strip()
        if len(raw) >= 2:
            digits_part = raw[1::2]  # every 2nd char = digit (LSD-first)
            carries_part = raw[0::2]  # every 2nd char = carry
            result = digits_part[::-1]  # reverse to normal order
            if carries_part and carries_part[-1] == '1':
                result = '1' + result
            display = result.lstrip('0') or '0'
        else:
            display = raw
    else:
        display = output
    
    expected = str(eval(prompt.replace('=', '')))
    correct = display == expected
    return display, expected, correct, raw

problems = ['12+34=', '99+1=', '45+67=', '7+8=', '123+456=']
print(f'{"Problem":<12} {"Predicted":<12} {"Expected":<12} {"Scratchpad":<16} {"Correct"}')
print('-' * 64)
for p in problems:
    display, expected, correct, raw = test_problem(p)
    mark = '✅' if correct else '❌'
    print(f'{p:<12} {display:<12} {expected:<12} {raw:<16} {mark}')

## 3. Inspect the Scratchpad Format

The V2 combined carry-digit format encodes each column as a single 2-character
token. Let's visualize the carry propagation across columns.

In [ ]:
def visualize_scratchpad(prompt):
    """Show the scratchpad breakdown for a problem."""
    output = model.generate(tok, prompt, max_new_tokens=20, temperature=0.0)
    
    if '=' in output:
        raw = output.split('=', 1)[-1].replace('<EOS>', '').replace('<PAD>', '').strip()
        if len(raw) >= 2:
            print(f'Problem: {prompt}')
            print(f'Raw scratchpad: {raw}')
            print()
            cols = len(raw) // 2
            print(f'Column-by-column (LSD → MSD):')
            col_names = ['ones', 'tens', 'hundreds', 'thousands', 'ten-thousands']
            for i in range(cols):
                carry = raw[i*2]
                digit = raw[i*2 + 1]
                name = col_names[i] if i < len(col_names) else f'col{i}'
                arrow = '→' if carry == '1' else ' '
                print(f'  {name}: carry_in={carry} {arrow} digit={digit}')
            
            # Final result
            digits = raw[1::2][::-1]
            if len(raw) >= 2 and raw[-2] == '1':
                digits = '1' + digits
            result = digits.lstrip('0') or '0'
            expected = str(eval(prompt.replace('=', '')))
            print(f'\nResult: {result} (expected: {expected})')
            print(f'Correct: {"✅" if result == expected else "❌"}')

visualize_scratchpad('47+89=')

## 4. Batch Evaluation

Evaluate the model's accuracy across different digit lengths.

In [ ]:
import random

def evaluate_accuracy(digits=1, samples=50):
    correct = 0
    for _ in range(samples):
        a = random.randint(10**(digits-1), 10**digits - 1) if digits > 1 else random.randint(1, 9)
        b = random.randint(10**(digits-1), 10**digits - 1) if digits > 1 else random.randint(1, 9)
        prompt = f'{a}+{b}='
        display, expected, correct_flag, _ = test_problem(prompt)
        if correct_flag:
            correct += 1
    return correct / samples * 100

print('Per-digit accuracy:')
print(f'{"Digits":<10} {"Accuracy":<12}')
print('-' * 22)
for d in [1, 2]:
    acc = evaluate_accuracy(d, 30)
    bar = '█' * int(acc / 5) + '░' * (20 - int(acc / 5))
    print(f'{d}-digit    {acc:>5.1f}%     {bar}')

## 5. Continual Learning (EWC)

Tabula Rasa uses **Online Elastic Weight Consolidation** to learn multiple
tasks sequentially without forgetting. The sleep cycle periodically:
1. Samples high-surprise experiences from the Hippocampus
2. Computes Fisher information on replay data
3. Merges into the running EWC total
4. Trains with an EWC penalty to consolidate without forgetting

In [ ]:
print('To run a sleep cycle:')
print('  python3 egefalos/sleep_cycle.py --model specialists/math/add/best.pt --daemon')
print()
print('To train a new specialist with EWC:')
print('  python3 train_specialist.py sub --ewc --steps 15000')
print()
print('To run Socratic self-improvement:')
print('  python3 train_specialist.py add --socratic --socratic-steps 500')

## 6. Dashboard & Visualization

Start the web dashboard for interactive exploration:

```bash
python3 api_server.py
# Open http://localhost:8000/ in your browser
```

Dashboard views include:
- **Training Monitor** — real-time loss/accuracy charts
- **Attention Flow** — animated carry propagation visualization
- **Fisher Heatmap** — EWC parameter importance (free vs locked)
- **Token Inspector** — vocabulary browser with encode/decode demo
- **System Status** — architecture whitepaper with all components

## 7. Docker Deployment

```bash
docker compose up -d
```

This starts:
1. **api-server** (port 8000) — dashboard + API + training
2. **router** (port 8002) — specialist routing
3. **sleep-cycle** — periodic EWC consolidation daemon

## Key References

- [README.md](README.md) — full project documentation
- [ROADMAP.md](ROADMAP.md) — project status and future plans
- [CONTRIBUTING.md](CONTRIBUTING.md) — developer guide
- [QUICKSTART.md](QUICKSTART.md) — CLI quickstart guide